# Track B — ajuste final no Colab

Este notebook executa **um único ajuste** do BERTimbau original em todos os
1.877 textos de desenvolvimento. Duração congelada: 318 passos, aproximadamente
2,69 épocas; semente 42. Ele não repete os três experimentos anteriores.

Selecione GPU (T4, se disponível) e execute as células em ordem.
Envie **track_b_final_package_v1.zip** na primeira célula.
O modelo salvo inclui pesos e tokenizer; o ZIP final terá aproximadamente
400–450 MB. Checkpoints continuam em uma nova pasta do Drive.


In [ ]:
from pathlib import Path
import hashlib, io, json, subprocess, sys, zipfile
from google.colab import files

uploaded = files.upload()
names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(names) != 1:
    raise ValueError('Envie somente o ZIP do ajuste final nesta célula.')
extract_root = Path('/content/pt_variant_final')
extract_root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[names[0]])) as archive:
    for item in archive.infolist():
        path = (extract_root / item.filename).resolve()
        if not path.is_relative_to(extract_root.resolve()):
            raise ValueError('Caminho inválido no ZIP.')
    archive.extractall(extract_root)
PACKAGE_ROOT = extract_root / 'track_b_final'
manifest = json.loads((PACKAGE_ROOT / 'package_manifest.json').read_text())
for name, expected in manifest['sha256'].items():
    if hashlib.sha256((PACKAGE_ROOT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f'Arquivo alterado: {name}')
selection = json.loads((PACKAGE_ROOT / 'selection.json').read_text())
print('Corpus:', manifest['rows'], 'textos; passos finais:', selection['final_protocol']['max_steps'])


## Preparar GPU e salvar no Drive
Se a sessão anterior já está aberta, pode usar a mesma conta e GPU.
O treino salva em **MyDrive/pt-variant-track-b/final_colab_v1**.
As pastas dos experimentos anteriores continuam disponíveis.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(PACKAGE_ROOT / 'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-c',
    "import torch; assert torch.cuda.is_available(), 'Selecione GPU no Colab'; "
    "print('GPU:', torch.cuda.get_device_name(0))"], check=True)
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_ROOT = Path('/content/drive/MyDrive/pt-variant-track-b/final_colab_v1')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## Treinar uma vez e validar a exportação
A duração foi escolhida apenas a partir dos checkpoints de desenvolvimento.
O script recomeça do encoder original com cabeça nova e treina todos os textos.
Se houver interrupção, repita esta célula: o último checkpoint será retomado.
Se já terminou, a célula apenas verifica o artefato salvo.


In [ ]:
command = [sys.executable, str(PACKAGE_ROOT / 'train_track_b_final.py'),
           '--package-root', str(PACKAGE_ROOT), '--output-dir', str(OUTPUT_ROOT)]
subprocess.run(command, check=True)
subprocess.run(command + ['--verify-only'], check=True)
final_manifest = json.loads((OUTPUT_ROOT / 'artifact/manifest.json').read_text())
print('Treino final concluído:', final_manifest['rows_fit'], 'textos')
print('Passos:', final_manifest['global_steps'])
print('Tempo de treino:', round(final_manifest['training_seconds_this_session'], 1), 'segundos')
print('Inferência CPU após recarregar: verificada')


## Exportar modelo completo e recibo pequeno
O ZIP grande contém apenas o artefato final: pesos, tokenizer, seleção e manifesto.
Ele não inclui dados de treino nem otimizadores. O recibo pequeno ajuda a verificar
a conclusão antes de transferir o modelo. Baixe os dois arquivos para Downloads.


In [ ]:
artifact = OUTPUT_ROOT / 'artifact'
archive_path = Path('/content/track_b_final_model_v1.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(artifact.rglob('*')):
        if path.is_file():
            archive.write(path, arcname='track_b_final_v1/' + path.relative_to(artifact).as_posix())
receipt_path = Path('/content/track_b_final_receipt_v1.json')
receipt = {'manifest': final_manifest,
           'artifact_zip_sha256': hashlib.sha256(archive_path.read_bytes()).hexdigest(),
           'artifact_zip_bytes': archive_path.stat().st_size}
receipt_path.write_text(json.dumps(receipt, ensure_ascii=False, indent=2))
print('Modelo completo:', round(archive_path.stat().st_size / 1024**2, 1), 'MiB')
files.download(str(receipt_path))
files.download(str(archive_path))
